In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
import numpy as np
import pandas as pd

In [2]:
#aggiornata la lettura a partire da index 0, saltando la colonna index
train_dataset = pd.read_csv('../../assets/seeds_11/train_11.csv', index_col=0) 
val_dataset = pd.read_csv('../../assets/seeds_11/val_11.csv', index_col=0) 
test_dataset = pd.read_csv('../../assets/seeds_11/test_11.csv', index_col=0) 
train_dataset.head()
train_dataset.info()
train_dataset.describe()

<class 'pandas.DataFrame'>
Index: 87447 entries, 76152 to 106757
Columns: 513 entries, 0 to label
dtypes: float64(512), int64(1)
memory usage: 342.9 MB


,0,1,2,3,4,5,6,7,8,9,...,503,504,505,506,507,508,509,510,511,label
count,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,...,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000,87447.000000
mean,0.679764,0.849554,0.445289,2.223606,0.402982,0.150932,0.787322,0.950320,1.445799,0.268028,...,1.291778,0.069529,1.362511,0.855332,0.747545,0.792669,0.943633,1.012457,1.951236,1.678731
std,0.536190,0.469740,0.377183,0.651790,0.390675,0.247751,0.378588,0.581192,0.565572,0.327745,...,0.647184,0.128277,0.751653,0.431239,0.547470,0.443908,0.750329,0.491377,0.821838,1.357813
min,0.000000,0.000000,0.000000,0.091206,0.000000,0.000000,0.000000,0.000000,0.007696,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.249447,0.493474,0.141849,1.773298,0.123679,0.001104,0.515237,0.516021,1.033248,0.047428,...,0.807376,0.000866,0.818487,0.536350,0.331734,0.466622,0.358371,0.652813,1.370411,0.000000
50%,0.569162,0.814180,0.364473,2.215324,0.297666,0.040499,0.760290,0.884864,1.406150,0.152698,...,1.232063,0.020850,1.254919,0.789990,0.645979,0.717027,0.770772,0.968429,1.942684,2.000000
75%,0.996849,1.155845,0.659865,2.653792,0.559518,0.190120,1.021999,1.302694,1.811211,0.367767,...,1.714781,0.079786,1.779154,1.105982,1.044386,1.040224,1.353090,1.318304,2.511128,3.000000
max,4.149904,3.961852,3.237006,5.986111,4.305758,2.643863,3.651450,4.827925,4.069973,3.734715,...,4.894304,2.531964,5.641622,3.502565,4.887534,3.861927,5.971847,4.277090,5.901646,3.000000


In [3]:
def create_model_pipeline(n_components):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n_components)),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=(64,),  # 1 hidden layer
            activation='relu',
            solver='adam',
            learning_rate_init=1e-3,
            max_iter=200,
            random_state=42
        ))
    ])

In [4]:

X_train = train_dataset.drop(columns=['label'])
y_train = train_dataset['label']
X_val = val_dataset.drop(columns=['label'])
y_val = val_dataset['label']
X_test = test_dataset.drop(columns=['label'])
y_test = test_dataset['label']


In [ ]:
from sklearn.metrics import f1_score
metric_results_f1_score = {}

for k in [32, 16, 8, 4]:
    model = create_model_pipeline(k)
    
    # 1. training
    model.fit(X_train, y_train)
    
    # 2. validation evaluation
    y_val_pred = model.predict(X_val)    
    f1_score_val = f1_score(y_val, y_val_pred, average='macro')
    metric_results_f1_score[k] = f1_score_val # Per ogni componente PCA

print(metric_results_f1_score)
'''{32: 0.8177863013380223, 16: 0.7582667908231424, 8: 0.6447482140403076, 4: 0.5187821722378831}
'''

{32: 0.8177863013380223, 16: 0.7582667908231424, 8: 0.6447482140403076, 4: 0.5187821722378831}


In [6]:
from sklearn.metrics import roc_auc_score


metric_results_auroc = {}

for k in [32, 16, 8, 4]:
    model = create_model_pipeline(k)
    
    model.fit(X_train, y_train)
    
    y_val_proba = model.predict_proba(X_val)
    
    auroc = roc_auc_score(
        y_val,
        y_val_proba,
        multi_class='ovr',
        average='macro'
    )
    
    metric_results_auroc[k] = auroc

    

In [ ]:
print(metric_results_auroc)
''' {32: 0.9697901173189223, 16: 0.9544475940486782, 8: 0.9229350335759114, 4: 0.8783457431738217}
'''

{32: 0.9697901173189223, 16: 0.9544475940486782, 8: 0.9229350335759114, 4: 0.8783457431738217}


In [ ]:
from sklearn.metrics import balanced_accuracy_score


metric_results_bal_acc = {}

for k in [32, 16, 8, 4]:
    model = create_model_pipeline(k)
    
    model.fit(X_train, y_train)
    
    y_val_pred = model.predict(X_val)
    
    bal_acc = balanced_accuracy_score(y_val, y_val_pred)
    
    metric_results_bal_acc[k] = bal_acc

print(metric_results_bal_acc)

"{32: 0.8001995096793855, 16: 0.7461754032177256, 8: 0.630554846447248, 4: 0.5280347503078432}"

{32: 0.8001995096793855, 16: 0.7461754032177256, 8: 0.630554846447248, 4: 0.5280347503078432}
